In [ ]:
# Import des bibliothèques nécessaires
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import getpass
from os import getenv
import oracledb
import warnings

In [ ]:
# Configuration pour éviter les avertissements
warnings.filterwarnings('ignore')

In [ ]:
# Configuration de seaborn pour de meilleurs graphiques
sns.set(style="whitegrid")

In [ ]:
# Nécessaire pour éviter les problèmes de session
class Connexion(object):
    def __init__(self, login, password):
        self.conn = oracledb.connect(
            user=login,
            password=password,
            host="oracle.iut-orsay.fr",
            port=1521,
            sid="etudom",
        )
        self.conn.autocommit = False

    def __enter__(self):
        self.conn.autocommit = False
        return self.conn

    def __exit__(self, *args):
        self.conn.close()

In [ ]:
# La fonction ci-dessous est à utiliser pour exécuter une requête et stocker les résultats dans un dataframe Pandas sans afficher d’alerte.
# Vous pouvez vous en inspirer pour créer vos propres fonctions.
def requete_vers_dataframe(connexion_data, requete, valeurs = None):
    with Connexion(login=connexion_data['login'], password=connexion_data['password']) as connexion:
        warnings.simplefilter(action='ignore', category=UserWarning)
        if valeurs is not None:
            df = pd.read_sql(requete, connexion, params=valeurs)
        else:
            df = pd.read_sql(requete, connexion)
        warnings.simplefilter("always") 
        return df

def requete_insert(connexion_data, requete, valeurs = None):
    with Connexion(login=connexion_data['login'], password=connexion_data['password']) as connexion:
        warnings.simplefilter(action='ignore', category=UserWarning)
        warnings.simplefilter("always")
        if valeurs is not None:
            df = pd.read_sql(requete, connexion, params=valeurs)
        else:
            df = pd.read_sql(requete, connexion)
        warnings.simplefilter("always")

In [ ]:
# Saisir ci-dessous la plateforme qui vous a été attribuée. Cela correspond au nomPlateforme dans la table PLATEFORME de la base de données
# Par exemple NOM_PLATEFORME = "Switch"
NOM_PLATEFORME = "Commodore C64/128/MAX"
# Saisir ci-dessous le login court de la base utilisée pour votre carnet
SCHEMA = '"AMETIN"'

In [ ]:
# On demande à l'utilisateur son login et mot de passe pour pouvoir accéder à la base de données
if getenv("DB_LOGIN") is None:
    login = input("Login : ")
else:
    login = getenv("DB_LOGIN")
if getenv("DB_PASS") is None:
    password = getpass.getpass("Mot de passe : ")
else:
    password = getenv("DB_PASS")
conn = {'login': login, 'password': password}

In [ ]:
# Traitement des CLOB
oracledb.defaults.fetch_lobs = False

In [ ]:
# On vérifie que l'utilisateur est bien connecté à la base de données, que le schéma est bon, et qu'on trouve la bonne édition des JO
data = requete_vers_dataframe(conn, f"SELECT * FROM {SCHEMA}.PLATEFORME WHERE nomPlateforme LIKE (:libelle)",{"libelle":NOM_PLATEFORME})
id_plateforme = int(data.IDPLATEFORME.iloc[0])
print(f"Identifiant de la plateforme : {id_plateforme}")

# Analyse statistique des données sur une plateforme

## Présentation générale de la plateforme Commodore C64/128/MAX

### Contexte historique

In [ ]:
data = f"""
SELECT 
    TO_CHAR(MIN(d.DATESORTIE), 'YYYY') AS annee_debut,
    TO_CHAR(MAX(d.DATESORTIE), 'YYYY') AS annee_fin,
    COUNT(DISTINCT d.IDJEU) AS total_jeux
FROM {SCHEMA}.DATESORTIE d
WHERE d.IDPLATEFORME = :id_plateforme
"""
historique = requete_vers_dataframe(conn, data, {'id_plateforme': id_plateforme})

print(f"""
**Commodore C64/128/MAX (1982-1994)**
- Première génération de la famille Commodore 64
- Commercialisée de {historique['ANNEE_DEBUT'][0]} à {historique['ANNEE_FIN'][0]}
- {historique['TOTAL_JEUX'][0]} jeux recensés
- Source : Base de données IGDB, MobyGames
""")

### Répartition des jeux par genre

In [ ]:
# Répartition par genre
requete = f"""
SELECT 
    g.NOMGENRE AS genre,
    COUNT(DISTINCT d.IDJEU) AS nb_jeux,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {SCHEMA}.DATESORTIE WHERE IDPLATEFORME = :id), 1) AS pourcentage
FROM {SCHEMA}.DATESORTIE d
INNER JOIN {SCHEMA}.GENREJEU gj ON d.IDJEU = gj.IDJEU
INNER JOIN {SCHEMA}.GENRE g ON gj.IDGENRE = g.IDGENRE
WHERE d.IDPLATEFORME = :id
GROUP BY g.NOMGENRE
ORDER BY nb_jeux DESC
"""
genres = requete_vers_dataframe(conn, requete, {'id': id_plateforme})

# Visualisation
plt.figure(figsize=(12,6))
sns.barplot(x='POURCENTAGE', y='GENRE', data=genres.head(10), palette='viridis')
plt.title("Top 10 des genres sur Commodore C64 (en % du total)")
plt.xlabel("Pourcentage des jeux")
plt.tight_layout()
plt.show()

### Évolution du nombre de jeux publiés par année

In [ ]:
# Requête pour l'évolution annuelle
requete = f"""
SELECT 
    EXTRACT(YEAR FROM d.DATESORTIE) AS annee,
    COUNT(d.IDJEU) AS nombre_jeux
FROM {SCHEMA}.DATESORTIE d
WHERE d.IDPLATEFORME = :id_plateforme
GROUP BY EXTRACT(YEAR FROM d.DATESORTIE)
ORDER BY annee
"""
evolution = requete_vers_dataframe(conn, requete, {'id_plateforme': id_plateforme})

# Graphique
plt.figure(figsize=(12, 6))
plt.plot(evolution['ANNEE'], evolution['NOMBRE_JEUX'], marker='o')
plt.title(f"Évolution du nombre de jeux publiés par année sur {NOM_PLATEFORME}")
plt.xlabel("Année")
plt.ylabel("Nombre de jeux")
plt.grid(True)
plt.show()

## Comparaison avec d'autres plateformes

### Classement des plateformes par nombre de jeux

In [ ]:
requete = f"""
SELECT 
    p.NOMPLATEFORME,
    COUNT(DISTINCT d.IDJEU) AS nombre_jeux
FROM {SCHEMA}.PLATEFORME p
LEFT JOIN {SCHEMA}.DATESORTIE d ON p.IDPLATEFORME = d.IDPLATEFORME
GROUP BY p.NOMPLATEFORME
HAVING COUNT(DISTINCT d.IDJEU) > 0
ORDER BY nombre_jeux DESC
"""
comparaison_plateformes = requete_vers_dataframe(conn, requete)

# On garde seulement les 20 premières plateformes pour une meilleure lisibilité
top_plateformes = comparaison_plateformes.head(20)

# Graphique
plt.figure(figsize=(12, 10))
ax = sns.barplot(x='NOMBRE_JEUX', y='NOMPLATEFORME', 
                 data=top_plateformes, 
                 hue='NOMPLATEFORME',
                 palette='rocket',
                 legend=False)

# Ajout des valeurs sur les barres
for i, (value, name) in enumerate(zip(top_plateformes['NOMBRE_JEUX'], top_plateformes['NOMPLATEFORME'])):
    ax.text(value + 50, i, f'{value:,}', va='center', fontsize=10)

plt.title("Top 20 des plateformes par nombre de jeux", pad=20, fontsize=14)
plt.xlabel("Nombre de jeux", labelpad=10)
plt.ylabel("")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Affichage du classement complet sous forme de tableau
print("Classement complet des plateformes par nombre de jeux :")
display(comparaison_plateformes.style.background_gradient(cmap='rocket_r', subset=['NOMBRE_JEUX']))

## Analyse d'une compagnie spécifique

In [ ]:
# Le nom de la compagnie analysé
NOM_COMPAGNIE = "System 3 Software"
id_compagnie = requete_vers_dataframe(conn,f"SELECT IDCOMPAGNIE FROM {SCHEMA}.COMPAGNIE WHERE NOMCOMPAGNIE = :nom",{'nom': NOM_COMPAGNIE})['IDCOMPAGNIE'][0]

### Contributions sur la plateforme Commodore C64/128/MAX

In [ ]:
requete = f"""
SELECT 
    j.TITREJEU AS titre,
    TO_CHAR(d.DATESORTIE, 'YYYY') AS annee,
    LISTAGG(g.NOMGENRE, ', ') WITHIN GROUP (ORDER BY g.NOMGENRE) AS genres
FROM {SCHEMA}.JEU j
INNER JOIN {SCHEMA}.DATESORTIE d ON j.IDJEU = d.IDJEU
INNER JOIN {SCHEMA}.COMPAGNIEJEU cj ON j.IDJEU = cj.IDJEU
LEFT JOIN {SCHEMA}.GENREJEU gj ON j.IDJEU = gj.IDJEU
LEFT JOIN {SCHEMA}.GENRE g ON gj.IDGENRE = g.IDGENRE
WHERE d.IDPLATEFORME = :id_plateforme
AND cj.IDCOMPAGNIE = :id_compagnie
AND cj.ESTDEVELOPPEUR = 1
GROUP BY j.TITREJEU, TO_CHAR(d.DATESORTIE, 'YYYY')
ORDER BY annee
"""

# Paramètres
params = {
    'id_plateforme': int(id_plateforme),
    'id_compagnie': int(id_compagnie)
}

jeux_compagnie = requete_vers_dataframe(conn, requete, params)

if not jeux_compagnie.empty:
    print(f"Jeux développés par {NOM_COMPAGNIE} ({len(jeux_compagnie)} titres):")
    
    # Affichage
    display(jeux_compagnie.style
           .background_gradient(subset=['ANNEE'], cmap='YlOrRd')
           .set_caption("Liste complète des jeux"))
    
    # Visualisation par année
    plt.figure(figsize=(10, 5))
    sns.countplot(x='ANNEE', data=jeux_compagnie)
    plt.title(f"Activité de développement par année\n{NOM_COMPAGNIE}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print(f"Aucun jeu trouvé pour {NOM_COMPAGNIE} sur {NOM_PLATEFORME}")

In [ ]:
requete = f"""
SELECT 
    c.NOMCOMPAGNIE,
    COUNT(DISTINCT d.IDJEU) AS nb_jeux,
    MIN(EXTRACT(YEAR FROM d.DATESORTIE)) AS premiere_annee,
    MAX(EXTRACT(YEAR FROM d.DATESORTIE)) AS derniere_annee
FROM {SCHEMA}.COMPAGNIE c
INNER JOIN {SCHEMA}.COMPAGNIEJEU cj ON c.IDCOMPAGNIE = cj.IDCOMPAGNIE
INNER JOIN {SCHEMA}.DATESORTIE d ON cj.IDJEU = d.IDJEU
WHERE d.IDPLATEFORME = :id_plateforme
AND cj.ESTDEVELOPPEUR = 1
GROUP BY c.NOMCOMPAGNIE
HAVING COUNT(DISTINCT d.IDJEU) >= 2
ORDER BY nb_jeux DESC
"""

compagnies = requete_vers_dataframe(conn, requete, {'id_plateforme': int(id_plateforme)})

if not compagnies.empty:
    print(f"\nComparaison avec {len(compagnies)} autres développeurs:")
    
    # Position de System 3 Software
    rang = compagnies[compagnies['NOMCOMPAGNIE'] == NOM_COMPAGNIE].index[0] + 1
    print(f"- {NOM_COMPAGNIE} est #{rang} avec {compagnies.loc[compagnies['NOMCOMPAGNIE'] == NOM_COMPAGNIE, 'NB_JEUX'].values[0]} jeux")
    
    # Visualisation
    plt.figure(figsize=(12, 6))
    sns.barplot(x='NB_JEUX', y='NOMCOMPAGNIE', 
                data=compagnies.head(10),
                palette='viridis')
    plt.title("Top 10 des développeurs sur C64")
    plt.xlabel("Nombre de jeux")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("Données insuffisantes pour comparaison")

In [ ]:
if not jeux_compagnie.empty and 'GENRES' in jeux_compagnie.columns:
    # Expansion des genres multiples
    genres = jeux_compagnie['GENRES'].str.split(', ', expand=True).stack()
    
    print(f"\nAnalyse des genres pour {NOM_COMPAGNIE}:")
    genre_counts = genres.value_counts()
    
    # Tableau
    display(genre_counts.to_frame('Nombre de jeux').transpose())
    
    # Graphique
    plt.figure(figsize=(10, 4))
    genre_counts.plot(kind='barh', color='darkcyan')
    plt.title("Répartition par genre")
    plt.xlabel("Nombre de jeux")
    plt.tight_layout()
    plt.show()

## Perspectives d'avenir

In [ ]:
requete = f"""
SELECT 
    p.NOMPLATEFORME,
    EXTRACT(YEAR FROM MIN(d.DATESORTIE)) AS premiere_annee,
    COUNT(DISTINCT d.IDJEU) AS nb_jeux
FROM {SCHEMA}.DATESORTIE d
INNER JOIN {SCHEMA}.PLATEFORME p ON d.IDPLATEFORME = p.IDPLATEFORME
INNER JOIN {SCHEMA}.COMPAGNIEJEU cj ON d.IDJEU = cj.IDJEU
WHERE cj.IDCOMPAGNIE = :id_compagnie
GROUP BY p.NOMPLATEFORME
ORDER BY premiere_annee
"""

# Exécution avec paramètre converti
evolution = requete_vers_dataframe(conn, requete, {'id_compagnie': int(id_compagnie)})

# Affichage sécurisé
if not evolution.empty:
    print("=== ÉVOLUTION MULTI-PLATEFORMES ===")
    display(evolution.style
           .format({'PREMIERE_ANNEE': '{:.0f}'})
           .background_gradient(subset=['NB_JEUX'], cmap='Greens'))
else:
    print("Aucune donnée d'évolution trouvée")